In [1]:
from google.colab import files
uploaded = files.upload()

Saving listings.csv to listings (1).csv


In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("listings.csv")

# Drop rows with missing target
df = df.dropna(subset=["price"])

# Fill missing features
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("missing")
    else:
        df[col] = df[col].fillna(df[col].median())

# One-hot encode categorical columns
df = pd.get_dummies(df, drop_first=True)

# Features & target
X = df.drop("price", axis=1)
y = df["price"]

print("Preprocessing done. Number of features:", X.shape[1])


Preprocessing done. Number of features: 29707


In [2]:
from sklearn.model_selection import train_test_split

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train-test split done.")
print("Training samples:", X_train.shape[0], "Testing samples:", X_test.shape[0])


Train-test split done.
Training samples: 17167 Testing samples: 4292


In [3]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression

# Base models
base_models = [
    ("rf", RandomForestRegressor(n_estimators=30, random_state=42)),  # fewer trees for speed
    ("gbr", GradientBoostingRegressor(n_estimators=30, random_state=42))
]

# Meta model
meta_model = LinearRegression()

# Stacking regressor
stack_reg = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model
)

print("Models defined.")


Models defined.


In [4]:
# Train
stack_reg.fit(X_train, y_train)

print("Training done.")


Training done.


In [5]:
from sklearn.metrics import mean_squared_error

# Predict
y_pred = stack_reg.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
print("Stacking Regressor MSE:", mse)


Stacking Regressor MSE: 1796631.3288792924


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error

# Load dataset
df = pd.read_csv("listings.csv")

# Drop rows with missing target
df = df.dropna(subset=["price"])

# Split features & target
X = df.drop("price", axis=1)
y = df["price"]

# Identify categorical & numerical columns
cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessor: numeric → median + scale, categorical → fill missing + one-hot
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ]
)

# Base models
base_models = [
    ("rf", RandomForestRegressor(n_estimators=30, random_state=42)),
    ("gbr", GradientBoostingRegressor(n_estimators=30, random_state=42))
]

# Meta model
meta_model = LinearRegression()

# Stacking regressor
stack_reg = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model
)

# Full pipeline: preprocessing + stacking
pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("stacking", stack_reg)
])

# Train
pipeline.fit(X_train, y_train)

# Predict & evaluate
y_pred = pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print("Pipeline Stacking Regressor MSE:", mse)
